# 81 字 · 第一阶段退化标定

本阶段：使用**原第 38 轮 best.pt** 对 25 档条件进行推理。共 **32,400 张**，来自原
`val_graph` 的 81 字 × 16 个共享清晰位移。σ=[0,1,1.5,2,2.5]，R=[64,48,32,24,16]，
始终输出 64×64。此阶段不更新模型、不读取原 test 图像，不决定正式训练难度。

**只需上传这个 Notebook 到 Colab，按顺序运行下面两格。** 源代码已嵌入，沿用原 Drive
中的数据 ZIP 和 `runs/baseline81_s42_v1/best.pt`。建议选择 GPU；CPU 也能推理，耗时更长。
不需要重新上传旧数据、模型或本地生成的标定集，也不需要运行旧 Notebook。

第①格固定 Pillow=12.3.0，生成与本地像素指纹一致的标定集。保留 Colab 已有 torch/torchvision；
推理环境会记录在新报告中，不假装与原训练环境完全相同。新代码和数据使用独立 `/content` 目录。
第②格先在本地完成推理和报告，再保存到 Drive 的新时间戳目录。原基线及其记录不被覆盖。

运行完成后，将第②格下载的 `hanzi81_phase1_results.zip` 回传，用于确定下一步人工盲认的档位。
旧模型掉分只说明对新退化的适应情况；仍需人工可读性标定与同分布基线重训。


In [ ]:
#@title ① 准备固定标定集（不会训练）
from pathlib import Path
import hashlib
import importlib.metadata
import json
import subprocess
import sys
import zipfile
from google.colab import drive

DRIVE_ROOT = Path("/content/drive/MyDrive/hanzi81")
WORK = Path("/content/hanzi81_degradation_phase1_v1")
EXPECTED_PIXELS = 'aafb1d4225faa7abe691fca4e9415bfc302e22c2e5beb230a723d528638426bc'
SOURCES = {'training/__init__.py': '"""81-character baseline training and reproducible data preparation."""\n', 'training/io_utils.py': '"""Small shared file helpers with no image-rendering dependency."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef verify_dataset(output):\n    output = Path(output)\n    manifest = json.loads((output / "manifest.json").read_text(encoding="utf-8"))\n    for name, expected in manifest["files"].items():\n        if digest(output / name) != expected:\n            raise ValueError(f"Dataset file hash mismatch: {name}")\n    return manifest\n', 'training/train.py': '"""ResNet18 baseline, with epoch-boundary checkpoints and explicit test gating."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport time\n\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nimport torchvision\nfrom torchvision.models import resnet18\n\n# This module deliberately does not import the renderer: training consumes the\n# immutable, checked dataset and never regenerates examples or uses test labels.\nfrom training.io_utils import digest, write_json, verify_dataset\n\n\nclass GlyphDataset(Dataset):\n    def __init__(self, directory, split, per_class=None):\n        with np.load(Path(directory) / f"{split}.npz", allow_pickle=False) as data:\n            self.images = data["images"]\n            self.labels = data["labels"]\n            self.condition_ids = data["condition_ids"]\n        if per_class:\n            indexes = np.concatenate([np.flatnonzero(self.labels == y)[:per_class]\n                                      for y in np.unique(self.labels)])\n            self.images, self.labels = self.images[indexes], self.labels[indexes]\n            self.condition_ids = self.condition_ids[indexes]\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, index):\n        image = torch.from_numpy(self.images[index].copy()).unsqueeze(0).float().div_(127.5).sub_(1)\n        return image, int(self.labels[index]), index\n\n\ndef model_for(n_classes):\n    model = resnet18(weights=None)\n    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)\n    nn.init.kaiming_normal_(model.conv1.weight, mode="fan_out", nonlinearity="relu")\n    model.maxpool = nn.Identity()\n    model.fc = nn.Linear(model.fc.in_features, n_classes)\n    return model\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True)\n\n\ndef environment(device):\n    return {"python": platform.python_version(), "torch": str(torch.__version__),\n            "torchvision": str(torchvision.__version__), "numpy": np.__version__,\n            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),\n            "device": str(device),\n            "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None}\n\n\ndef save_checkpoint(path, state):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    torch.save(state, temporary)\n    os.replace(temporary, path)\n\n\ndef loader_for(dataset, batch_size, device, seed=None):\n    # No online augmentation or worker RNG. A separate per-epoch generator makes\n    # the shuffle independent of evaluations and reproducible after resumption.\n    generator = torch.Generator().manual_seed(0 if seed is None else seed)\n    return DataLoader(dataset, batch_size=batch_size, shuffle=seed is not None,\n                      num_workers=0, pin_memory=device.type == "cuda", generator=generator)\n\n\ndef epoch_pass(model, loader, device, n_classes, optimizer=None, scaler=None):\n    training = optimizer is not None\n    model.train(training)\n    total_loss, n = 0.0, 0\n    matrix = np.zeros((n_classes, n_classes), dtype=np.int64)\n    all_predictions = np.empty(len(loader.dataset), dtype=np.int64)\n    with torch.set_grad_enabled(training):\n        for images, labels, indexes in loader:\n            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)\n            if training:\n                optimizer.zero_grad(set_to_none=True)\n            # Use AMP for training on CUDA; use fp32 for checkpoint selection.\n            with torch.autocast(device_type=device.type, enabled=training and device.type == "cuda"):\n                logits = model(images)\n                loss = nn.functional.cross_entropy(logits, labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite loss; checkpoint was not advanced")\n            if training:\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n            predicted = logits.detach().argmax(1).cpu().numpy()\n            truth = labels.cpu().numpy()\n            np.add.at(matrix, (truth, predicted), 1)\n            all_predictions[indexes.numpy()] = predicted\n            total_loss += loss.item() * len(labels)\n            n += len(labels)\n    denominator = matrix.sum(1)\n    metrics = {"loss": total_loss / n, "accuracy": float(np.trace(matrix) / n),\n               "macro_accuracy": float(np.mean(np.diag(matrix)[denominator > 0] / denominator[denominator > 0])),\n               "samples": n}\n    return metrics, matrix, all_predictions\n\n\ndef train(args, manifest, labels, device):\n    out = args.run_dir\n    out.mkdir(parents=True, exist_ok=True)\n    config = {\n        "mode": args.mode, "seed": args.seed, "epochs": args.epochs, "batch_size": args.batch_size,\n        "lr": args.lr, "weight_decay": args.weight_decay, "per_class": args.per_class,\n        "model": "resnet18-gray3x3-s1-no-maxpool-random", "loss": "cross_entropy",\n        "data_sha256": digest(args.data / "manifest.json"), "trainer_sha256": digest(__file__),\n        "io_sha256": digest(Path(__file__).with_name("io_utils.py")),\n        "environment": environment(device), "amp": device.type == "cuda",\n    }\n    if args.mode == "baseline" and args.per_class is not None:\n        raise ValueError("Per-class truncation is only allowed for smoke/overfit modes")\n    if not args.resume and any(out.iterdir()):\n        raise ValueError("Run directory is nonempty. Use --resume or choose a new run name.")\n    train_data = GlyphDataset(args.data, "train", args.per_class)\n    validation = train_data if args.mode == "overfit" else GlyphDataset(args.data, "val_select", args.per_class)\n    validation_loader = loader_for(validation, args.batch_size, device)\n    model = model_for(len(labels)).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=args.lr / 100)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    start, history, best_score = 0, [], (-math.inf, -math.inf)\n    if args.resume:\n        saved = torch.load(out / "last.pt", map_location="cpu", weights_only=True)\n        if saved["config"] != config:\n            raise ValueError("Resume configuration/environment/data/code changed. Restore the original settings or use a new run.")\n        model.load_state_dict(saved["model"])\n        optimizer.load_state_dict(saved["optimizer"])\n        scheduler.load_state_dict(saved["scheduler"])\n        scaler.load_state_dict(saved["scaler"])\n        torch.set_rng_state(saved["torch_rng"])\n        if device.type == "cuda":\n            torch.cuda.set_rng_state_all(saved["cuda_rng"])\n        start, history, best_score = saved["epoch"], saved["history"], tuple(saved["best_score"])\n        if not (out / "best.pt").exists():\n            raise ValueError("best.pt is missing; restore the complete run directory")\n        print(f"Resuming after completed epoch {start}", flush=True)\n    write_json(out / "config.json", config)\n    write_json(out / "history.json", history)\n    write_json(out / "labels.json", labels)\n    write_json(out / "data_manifest.json", manifest)\n    stop = min(args.epochs, start + args.stop_after) if args.stop_after else args.epochs\n    for epoch in range(start, stop):\n        beginning = time.perf_counter()\n        train_loader = loader_for(train_data, args.batch_size, device, seed=args.seed + epoch)\n        lr = optimizer.param_groups[0]["lr"]\n        train_metrics, _, _ = epoch_pass(model, train_loader, device, len(labels), optimizer, scaler)\n        val_metrics, _, _ = epoch_pass(model, validation_loader, device, len(labels))\n        scheduler.step()\n        record = {"epoch": epoch + 1, "lr": lr, "train": train_metrics,\n                  "validation_split": "train-subset" if args.mode == "overfit" else "val_select",\n                  "validation": val_metrics, "seconds_compute": time.perf_counter() - beginning}\n        history.append(record)\n        score = (val_metrics["macro_accuracy"], -val_metrics["loss"])\n        if score > best_score:\n            best_score = score\n            save_checkpoint(out / "best.pt", {"model": model.state_dict(), "config": config,\n                                               "epoch": epoch + 1, "metrics": val_metrics})\n        save_checkpoint(out / "last.pt", {\n            "model": model.state_dict(), "optimizer": optimizer.state_dict(),\n            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),\n            "epoch": epoch + 1, "config": config, "history": history, "best_score": best_score,\n            "torch_rng": torch.get_rng_state(),\n            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],\n        })\n        write_json(out / "history.json", history)\n        print(json.dumps(record, ensure_ascii=False), flush=True)\n        print(f"Checkpoint saved: {out / \'last.pt\'}; epoch wall time incl. saves: {time.perf_counter() - beginning:.1f}s", flush=True)\n    print(f"Completed {len(history)}/{args.epochs} epochs. Best macro accuracy: {best_score[0]:.4f}", flush=True)\n\n\ndef evaluate(args, manifest, label_data, device):\n    if args.split == "test" and not args.final_test:\n        raise ValueError("Test is reserved. Explicit --final-test is required after the protocol is frozen.")\n    checkpoint = torch.load(args.run_dir / "best.pt", map_location="cpu", weights_only=True)\n    if checkpoint["config"]["mode"] != "baseline":\n        raise ValueError("Only a baseline checkpoint can be used for development/final reports")\n    if checkpoint["config"]["data_sha256"] != digest(args.data / "manifest.json"):\n        raise ValueError("Checkpoint and dataset do not match")\n    if checkpoint["config"]["environment"] != environment(device):\n        raise ValueError("Evaluation environment differs from the training environment")\n    if checkpoint["config"]["trainer_sha256"] != digest(__file__):\n        raise ValueError("Evaluation code differs from the checkpoint source")\n    labels = label_data["characters"]\n    dataset = GlyphDataset(args.data, args.split)\n    model = model_for(len(labels)).to(device)\n    model.load_state_dict(checkpoint["model"])\n    metrics, matrix, prediction = epoch_pass(model, loader_for(dataset, args.batch_size, device), device, len(labels))\n    conditions = json.loads((args.data / "conditions.json").read_text(encoding="utf-8"))[args.split]\n    sigma = np.array([conditions[int(i)]["sigma"] for i in dataset.condition_ids])\n    metrics["by_sigma"] = {str(s): {"samples": int((sigma == s).sum()),\n        "accuracy": float(np.mean(prediction[sigma == s] == dataset.labels[sigma == s]))}\n        for s in sorted(set(sigma.tolist()))}\n    metrics.update({"split": args.split, "checkpoint_epoch": checkpoint["epoch"],\n                    "checkpoint_sha256": digest(args.run_dir / "best.pt")})\n    char_to_id = {c["char"]: c["class_id"] for c in labels}\n    pairs = []\n    for pair in label_data["pairs"]:\n        a, b = char_to_id[pair["a"]], char_to_id[pair["b"]]\n        pairs.append({"a": pair["a"], "b": pair["b"], "a_to_b": int(matrix[a, b]),\n                      "b_to_a": int(matrix[b, a]),\n                      "pair_error_rate": float((matrix[a, b] + matrix[b, a]) / (matrix[a].sum() + matrix[b].sum()))})\n    report_dir = args.run_dir / f"evaluation_{args.split}"\n    report_dir.mkdir(exist_ok=True)\n    np.save(report_dir / "confusion.npy", matrix)\n    write_json(report_dir / "metrics.json", metrics)\n    write_json(report_dir / "pairs.json", pairs)\n    with (report_dir / "predictions.csv").open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["index", "true_id", "true_char", "predicted_id", "predicted_char", "sigma", "dx", "dy"])\n        for index, (truth, pred) in enumerate(zip(dataset.labels, prediction)):\n            condition = conditions[int(dataset.condition_ids[index])]\n            writer.writerow([index, truth, labels[int(truth)]["char"], pred, labels[int(pred)]["char"],\n                             condition["sigma"], condition["dx"], condition["dy"]])\n    print(json.dumps(metrics, ensure_ascii=False, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=["train", "evaluate"])\n    parser.add_argument("--data", type=Path, required=True)\n    parser.add_argument("--run-dir", type=Path, required=True)\n    parser.add_argument("--mode", choices=["smoke", "overfit", "baseline"], default="baseline")\n    parser.add_argument("--epochs", type=int, default=40)\n    parser.add_argument("--batch-size", type=int, default=64)\n    parser.add_argument("--lr", type=float, default=0.001)\n    parser.add_argument("--weight-decay", type=float, default=0.0001)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--per-class", type=int)\n    parser.add_argument("--stop-after", type=int, help="Stop after N additional epochs without changing the planned schedule")\n    parser.add_argument("--resume", action="store_true")\n    parser.add_argument("--allow-cpu", action="store_true", help="Only for local verification")\n    parser.add_argument("--split", choices=["val_select", "val_graph", "test"], default="val_select")\n    parser.add_argument("--final-test", action="store_true")\n    args = parser.parse_args()\n    if min(args.epochs, args.batch_size) <= 0 or (args.per_class is not None and args.per_class <= 0):\n        parser.error("epochs, batch-size and per-class must be positive")\n    if args.stop_after is not None and args.stop_after <= 0:\n        parser.error("stop-after must be positive")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type == "cpu" and not args.allow_cpu:\n        raise RuntimeError("No CUDA GPU. In Colab select Runtime > Change runtime type > GPU, then reconnect.")\n    if device.type == "cpu":\n        torch.set_num_threads(2)\n    seed_everything(args.seed)\n    manifest = verify_dataset(args.data)\n    label_data = json.loads((args.data / "labels.json").read_text(encoding="utf-8"))\n    print(json.dumps(environment(device), indent=2), flush=True)\n    if args.command == "train":\n        train(args, manifest, label_data["characters"], device)\n    else:\n        evaluate(args, manifest, label_data, device)\n\n\nif __name__ == "__main__":\n    main()\n', 'training/degradation.py': '"""Phase 1: a fixed development-only blur/resolution sweep and frozen-model evaluation."""\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport platform\nimport random\n\nimport numpy as np\nimport PIL\nfrom PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageOps\n\nfrom training.io_utils import digest, write_json\n\nSIGMAS = [0, 1, 1.5, 2, 2.5]\nRESOLUTIONS = [64, 48, 32, 24, 16]\nSEED = 20261020\nBASE_MANIFEST_SHA256 = "6a2d237ae03ed219f76c2d4cf827eeb6b9c852c88c54a5a77d6ca34706d84f86"\nCHECKPOINT_SHA256 = "8d65412bf6065253ebce3ca5b0c542788b71e0da391cc335a815f980d828a106"\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef degrade(image, sigma, resolution):\n    """Sigma is in 64x64 pixels; retain the original renderer\'s white boundary."""\n    if sigma:\n        image = ImageOps.expand(image, border=16, fill=255).filter(\n            ImageFilter.GaussianBlur(sigma)).crop((16, 16, 80, 80))\n    if resolution != 64:\n        image = image.resize((resolution, resolution), Image.Resampling.LANCZOS)\n        image = image.resize((64, 64), Image.Resampling.BICUBIC)\n    return image.copy()\n\n\ndef select_conditions(conditions):\n    clear = [i for i, c in enumerate(conditions) if c["sigma"] == 0]\n    center = next(i for i in clear if conditions[i]["dx"] == conditions[i]["dy"] == 0)\n    rng = random.Random(SEED)\n    # Offline geometric balancing only; no model predictions enter selection.\n    for _ in range(10000):\n        ids = sorted([center] + rng.sample([i for i in clear if i != center], 15))\n        axes = [[conditions[i][axis] for i in ids] for axis in ("dx", "dy")]\n        if all(abs(sum(v) / len(v)) <= .25 and min(v) <= -2.5 and max(v) >= 2.5 for v in axes):\n            return ids\n    raise ValueError("Could not select 16 balanced, shared development conditions")\n\n\ndef generate(data, output):\n    data, output = Path(data), Path(output)\n    if output.exists():\n        raise FileExistsError(f"Use a new calibration directory: {output}")\n    if digest(data / "manifest.json") != BASE_MANIFEST_SHA256:\n        raise ValueError("Phase 1 requires the archived baseline81-v2 data manifest")\n    source_manifest = json.loads((data / "manifest.json").read_text(encoding="utf-8"))\n    # Only these development inputs are opened. Do not load train/test arrays.\n    for name in ("val_graph.npz", "conditions.json", "labels.json"):\n        if digest(data / name) != source_manifest["files"][name]:\n            raise ValueError(f"Changed source data: {name}")\n    label_data = json.loads((data / "labels.json").read_text(encoding="utf-8"))\n    conditions = json.loads((data / "conditions.json").read_text(encoding="utf-8"))["val_graph"]\n    ids = select_conditions(conditions)\n    with np.load(data / "val_graph.npz", allow_pickle=False) as source:\n        indexes = np.concatenate([np.flatnonzero(source["condition_ids"] == i) for i in ids])\n        base = source["images"][indexes]\n        labels = source["labels"][indexes]\n        source_condition_ids = source["condition_ids"][indexes]\n    np.testing.assert_array_equal(labels, np.tile(np.arange(81), 16))\n    grids = [{"grid_id": i, "sigma": s, "resolution": r}\n             for i, (s, r) in enumerate((s, r) for s in SIGMAS for r in RESOLUTIONS)]\n    images = np.stack([np.stack([np.asarray(degrade(Image.fromarray(x), g["sigma"], g["resolution"]))\n                                for x in base]) for g in grids])\n    np.testing.assert_array_equal(images[0], base)\n    audit = audit_pixels(images, labels)\n    if audit["blank_images"] or audit["cross_label_collisions"]:\n        raise ValueError(f"Invalid calibration pixels: {audit}")\n    output.mkdir(parents=True)\n    np.savez_compressed(output / "sweep.npz", images=images, labels=labels,\n                        source_indexes=indexes, source_condition_ids=source_condition_ids)\n    (output / "labels.json").write_bytes((data / "labels.json").read_bytes())\n    manifest = {\n        "schema": 1, "purpose": "development_calibration_only", "dataset": "degradation81-calibration-v1",\n        "source_split": "val_graph", "source_manifest_sha256": BASE_MANIFEST_SHA256,\n        "selection_seed": SEED, "source_condition_ids": ids,\n        "conditions": [{"source_condition_id": i, **conditions[i]} for i in ids],\n        "source_group_key": "baseline81-v2/val_graph/source_index; all 25 derivatives stay together",\n        "grids": grids, "shape": list(images.shape), "samples": int(images.shape[0] * images.shape[1]),\n        "characters": len(label_data["characters"]), "pairs": len(label_data["pairs"]),\n        "samples_per_character_per_grid": 16, "normalization": "uint8 / 127.5 - 1",\n        "pipeline": ["archived clear 64x64", "white-pad16 Gaussian(sigma in 64px units)",\n                     "Lanczos to R", "bicubic to 64; bypass both resizes when R=64"],\n        "runtime": {"python": platform.python_version(), "numpy": np.__version__, "Pillow": PIL.__version__},\n        "files": {name: digest(output / name) for name in ("sweep.npz", "labels.json")},\n        "pixel_sha256": hashlib.sha256(images.tobytes()).hexdigest(),\n        "model_evaluation": "pending_original_checkpoint", "human_readability": "not_measured",\n        "interpretation": "Related synthetic variants, not independent real glyphs. No training or final test.",\n    }\n    audit.update({"clean_identity": True, "source_split": "val_graph", "test_arrays_opened": False,\n                  "position_statistics": {a: {"mean": float(np.mean([conditions[i][a] for i in ids])),\n                      "min": min(conditions[i][a] for i in ids), "max": max(conditions[i][a] for i in ids)}\n                      for a in ("dx", "dy")}})\n    write_json(output / "audit.json", audit)\n    write_json(output / "manifest.json", manifest)\n    return manifest, audit\n\n\ndef audit_pixels(images, labels):\n    seen, collisions, duplicate_count = {}, [], 0\n    for grid_id, batch in enumerate(images):\n        for base_id, pixels in enumerate(batch):\n            key = hashlib.sha256(pixels.tobytes()).digest()\n            value = (int(labels[base_id]), grid_id, base_id)\n            if key in seen:\n                duplicate_count += 1\n                if seen[key][0] != value[0]:\n                    collisions.append({"first": seen[key], "second": value})\n            else:\n                seen[key] = value\n    edge = np.concatenate((images[:, :, 0, :], images[:, :, -1, :],\n                           images[:, :, :, 0], images[:, :, :, -1]), axis=-1)\n    return {"images": int(np.prod(images.shape[:2])), "unique_pixels": len(seen),\n            "same_label_duplicate_images": duplicate_count - len(collisions),\n            "cross_label_collisions": collisions,\n            "blank_images": int(np.sum(np.min(images, axis=(-2, -1)) == 255)),\n            "maximum_border_darkness": int(255 - edge.min()),\n            "images_with_border_darkness_ge_16": int(np.sum(255 - edge.min(axis=-1) >= 16)),\n            "note": "Pixel uniqueness is a data audit, not a readability or difficulty measurement."}\n\n\ndef make_gallery(directory, font_path):\n    directory = Path(directory)\n    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))\n    labels = json.loads((directory / "labels.json").read_text(encoding="utf-8"))\n    with np.load(directory / "sweep.npz", allow_pickle=False) as saved:\n        images, source_ids, truth = saved["images"], saved["source_condition_ids"], saved["labels"]\n    center = next(c["source_condition_id"] for c in manifest["conditions"] if c["dx"] == c["dy"] == 0)\n    by_char = {c["char"]: np.flatnonzero((source_ids == center) & (truth == c["class_id"]))[0]\n               for c in labels["characters"]}\n    preview = directory / "preview"\n    preview.mkdir(exist_ok=True)\n    font = ImageFont.truetype(str(font_path), 20)\n    small = ImageFont.truetype(str(font_path), 15)\n    for pair in labels["pairs"]:\n        sheet = Image.new("RGB", (850, 650), "#f4f6fa")\n        draw = ImageDraw.Draw(sheet)\n        draw.text((20, 12), f"{pair[\'a\']} / {pair[\'b\']} · 25 档退化标定", font=font, fill="#182c44")\n        draw.text((20, 44), "每字原尺寸 64×64；左／右顺序固定。此页有标签，不是人工盲认测试。", font=small, fill="#526175")\n        draw.text((20, 70), pair["region"], font=small, fill="#526175")\n        for g in manifest["grids"]:\n            gi = g["grid_id"]\n            x, y = 20 + (gi % 5) * 166, 110 + (gi // 5) * 106\n            draw.text((x, y), f"σ={g[\'sigma\']:g} / R={g[\'resolution\']}", font=small, fill="#526175")\n            for j, char in enumerate((pair["a"], pair["b"])):\n                sheet.paste(Image.fromarray(images[gi, by_char[char]]).convert("RGB"), (x + j * 72, y + 26))\n        sheet.save(preview / f"pair_{pair[\'pair_id\']:02d}.png")\n    options = "".join(f\'<option value="{p["pair_id"]:02d}">{p["pair_id"]:02d} · {p["a"]} / {p["b"]}</option>\'\n                      for p in labels["pairs"])\n    html = f\'\'\'<!doctype html><html lang="zh-CN"><meta charset="utf-8"><title>81 字退化标定</title>\n<style>body{{font:16px system-ui;background:#f4f6fa;color:#182c44;max-width:1100px;margin:30px auto;padding:0 16px}}\nselect,button{{font:inherit;padding:8px}}img{{display:block;margin-top:16px;max-width:100%}}.zoom{{max-width:none;width:1700px;image-rendering:pixelated}}</style>\n<h1>81 字 · 第一阶段退化标定</h1><p>25 档 × 81 字 × 16 个共享位移 = 32,400 张开发图。模型推理待在 Colab 执行；人类可读性尚未测量。</p>\n<p>以下展示居中条件。σ 按 64×64 像素计；R 是中间分辨率。全体类别共用条件，无关键笔画擦除。</p>\n<label>重点字对 <select id="pair">{options}</select></label> <button id="zoom">切换 2 倍像素放大</button>\n<img id="sheet" alt="字对在 25 档条件下的实际像素" src="pair_01.png">\n<script>document.getElementById(\'pair\').onchange=e=>document.getElementById(\'sheet\').src=\'pair_\'+e.target.value+\'.png\';\ndocument.getElementById(\'zoom\').onclick=()=>document.getElementById(\'sheet\').classList.toggle(\'zoom\');</script></html>\'\'\'\n    (preview / "index.html").write_text(html, encoding="utf-8")\n\n\ndef summarize(truth, prediction, losses, label_data):\n    n_classes = len(label_data["characters"])\n    matrix = np.zeros((n_classes, n_classes), dtype=np.int64)\n    np.add.at(matrix, (truth, prediction), 1)\n    count = matrix.sum(axis=1)\n    accuracy = np.diag(matrix) / count\n    char_to_id = {c["char"]: c["class_id"] for c in label_data["characters"]}\n    pairs = []\n    for p in label_data["pairs"]:\n        a, b = char_to_id[p["a"]], char_to_id[p["b"]]\n        pairs.append({"pair_id": p["pair_id"], "a": p["a"], "b": p["b"],\n                      "a_to_b": int(matrix[a, b]), "b_to_a": int(matrix[b, a]),\n                      "pair_error_rate": float((matrix[a, b] + matrix[b, a]) / (count[a] + count[b]))})\n    # Original S rule: for each source class choose its most frequent wrong target.\n    wrong = matrix.copy()\n    np.fill_diagonal(wrong, 0)\n    target = wrong.argmax(axis=1)\n    support = [{"source_id": i, "target_id": int(target[i]), "count": int(wrong[i, target[i]])}\n               for i in range(n_classes) if wrong[i, target[i]] >= 5]\n    return {"samples": len(truth), "errors": int(np.sum(truth != prediction)),\n            "accuracy": float(np.mean(truth == prediction)), "macro_accuracy": float(accuracy.mean()),\n            "loss": float(np.mean(losses)), "per_class_accuracy": accuracy.tolist(),\n            "supported_sources_ge5": support, "pairs": pairs}, matrix\n\n\ndef write_report(output, rows, wrong_examples):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    output = Path(output)\n    fig, ax = plt.subplots(figsize=(7.6, 4.6))\n    for resolution in RESOLUTIONS:\n        group = [r for r in rows if r["resolution"] == resolution]\n        ax.plot([r["sigma"] for r in group], [100 * r["macro_accuracy"] for r in group],\n                marker="o", label=f"R={resolution}")\n    ax.set(xlabel="Gaussian sigma (64x64 pixels)", ylabel="Macro accuracy (%)", ylim=(0, 101),\n           title="Frozen baseline / development calibration")\n    ax.grid(alpha=.25)\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(output / "accuracy_curves.png", dpi=160)\n    plt.close(fig)\n    table = "".join(f\'<tr><td>{r["sigma"]}</td><td>{r["resolution"]}</td><td>{r["macro_accuracy"]:.2%}</td>\'\n                    f\'<td>{r["errors"]}/{r["samples"]}</td><td>{len(r["supported_sources_ge5"])}</td></tr>\' for r in rows)\n    cards = []\n    for e in wrong_examples:\n        stream = io.BytesIO()\n        Image.fromarray(e["pixels"]).save(stream, format="PNG")\n        url = "data:image/png;base64," + base64.b64encode(stream.getvalue()).decode()\n        cards.append(f\'<figure><img src="{url}" width="64" height="64"><figcaption>{e["true"]} → {e["predicted"]}\'\n                     f\'<br>σ={e["sigma"]}, R={e["resolution"]}<br>source={e["source_index"]}</figcaption></figure>\')\n    html = f\'\'\'<!doctype html><html lang="zh-CN"><meta charset="utf-8"><title>退化标定结果</title>\n<style>body{{font:16px system-ui;max-width:1050px;margin:30px auto;padding:0 16px;color:#182c44}}td,th{{padding:7px 14px;border-bottom:1px solid #ddd}}\n.cards{{display:flex;flex-wrap:wrap}}figure{{margin:12px;width:145px}}figcaption{{font-size:13px}}img{{max-width:100%}}</style>\n<h1>第一阶段：旧模型退化扫描</h1><p>仅反映旧模型遇到新退化后的表现。尚未完成同分布重训或人类可读性标定，不能据此选定正式训练难度。</p>\n<img src="accuracy_curves.png" alt="准确率随模糊强度变化"><table><tr><th>σ</th><th>R</th><th>宏准确率</th><th>错误数</th><th>有≥5次支持的源类别</th></tr>{table}</table>\n<h2>错误样例</h2><p>按档位、来源顺序，每档最多 4 张；完整记录见 predictions.csv。此处不是盲认测试。</p><div class="cards">{\'\'.join(cards)}</div></html>\'\'\'\n    (output / "report.html").write_text(html, encoding="utf-8")\n\n\ndef evaluate(directory, checkpoint_path, output, batch_size=128, device_name="auto",\n             expected_checkpoint_sha256=CHECKPOINT_SHA256):\n    import torch\n    from training import train as T\n\n    directory, checkpoint_path, output = Path(directory), Path(checkpoint_path), Path(output)\n    if output.exists():\n        raise FileExistsError(f"Keep the previous report; use a new output directory: {output}")\n    if digest(checkpoint_path) != expected_checkpoint_sha256:\n        raise ValueError("Checkpoint does not match the archived epoch-38 baseline")\n    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))\n    for name, expected in manifest["files"].items():\n        if digest(directory / name) != expected:\n            raise ValueError(f"Calibration file changed: {name}")\n    label_data = json.loads((directory / "labels.json").read_text(encoding="utf-8"))\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if checkpoint["config"]["mode"] != "baseline" or checkpoint["config"]["data_sha256"] != manifest["source_manifest_sha256"]:\n        raise ValueError("Checkpoint is not the source-data baseline")\n    if checkpoint["config"]["trainer_sha256"] != digest(Path(T.__file__)):\n        raise ValueError("Restore the archived model implementation before evaluation")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device_name == "auto" else torch.device(device_name)\n    if device.type == "cpu":\n        torch.set_num_threads(2)\n    T.seed_everything(42)\n    model = T.model_for(len(label_data["characters"])).to(device)\n    model.load_state_dict(checkpoint["model"])\n    model.eval()\n    with np.load(directory / "sweep.npz", allow_pickle=False) as saved:\n        images, truth = saved["images"], saved["labels"]\n        source_indexes, source_ids = saved["source_indexes"], saved["source_condition_ids"]\n    output.mkdir(parents=True)\n    rows, matrices, wrong_examples = [], [], []\n    chars = [c["char"] for c in label_data["characters"]]\n    with (output / "predictions.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["grid_id", "sigma", "resolution", "source_index", "source_condition_id", "true_id",\n                         "true_char", "predicted_id", "predicted_char", "confidence", "true_probability_margin", "loss"])\n        for grid in manifest["grids"]:\n            batch_images = images[grid["grid_id"]]\n            predicted, losses, confidences, margins = [], [], [], []\n            with torch.inference_mode():\n                for start in range(0, len(truth), batch_size):\n                    x = torch.from_numpy(batch_images[start:start + batch_size].copy()).unsqueeze(1).float().div_(127.5).sub_(1).to(device)\n                    y = torch.as_tensor(truth[start:start + batch_size], device=device)\n                    logits = model(x)\n                    if not torch.isfinite(logits).all():\n                        raise FloatingPointError("Non-finite logits in calibration inference")\n                    loss = torch.nn.functional.cross_entropy(logits, y, reduction="none")\n                    probabilities = logits.softmax(1)\n                    confidence, pred = probabilities.max(1)\n                    true_probability = probabilities.gather(1, y[:, None]).squeeze(1)\n                    probabilities.scatter_(1, y[:, None], -1)\n                    margin = true_probability - probabilities.max(1).values\n                    predicted.extend(pred.cpu().tolist())\n                    losses.extend(loss.cpu().tolist())\n                    confidences.extend(confidence.cpu().tolist())\n                    margins.extend(margin.cpu().tolist())\n            predicted = np.asarray(predicted)\n            metrics, matrix = summarize(truth, predicted, losses, label_data)\n            row = {**grid, **metrics}\n            rows.append(row)\n            matrices.append(matrix)\n            for i, pred in enumerate(predicted):\n                writer.writerow([grid["grid_id"], grid["sigma"], grid["resolution"], int(source_indexes[i]),\n                                 int(source_ids[i]), int(truth[i]), chars[truth[i]], int(pred), chars[pred],\n                                 confidences[i], margins[i], losses[i]])\n            for i in np.flatnonzero(predicted != truth)[:4]:\n                wrong_examples.append({"pixels": batch_images[i], "true": chars[truth[i]], "predicted": chars[predicted[i]],\n                                       "sigma": grid["sigma"], "resolution": grid["resolution"], "source_index": int(source_indexes[i])})\n            print(f"[{grid[\'grid_id\'] + 1:02d}/25] sigma={grid[\'sigma\']:g} R={grid[\'resolution\']}: "\n                  f"accuracy={metrics[\'macro_accuracy\']:.2%}, errors={metrics[\'errors\']}/{metrics[\'samples\']}", flush=True)\n    write_json(output / "metrics.json", rows)\n    np.savez_compressed(output / "confusion.npz", matrices=np.stack(matrices))\n    with (output / "summary.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["grid_id", "sigma", "resolution", "samples", "errors", "macro_accuracy", "loss", "supported_sources_ge5"])\n        for r in rows:\n            writer.writerow([r[k] for k in ("grid_id", "sigma", "resolution", "samples", "errors", "macro_accuracy", "loss")]\n                            + [len(r["supported_sources_ge5"])])\n    with (output / "pairs.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        fields = ["grid_id", "sigma", "resolution", "pair_id", "a", "b", "a_to_b", "b_to_a", "pair_error_rate"]\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        for row in rows:\n            for pair in row["pairs"]:\n                writer.writerow({**{k: row[k] for k in ("grid_id", "sigma", "resolution")}, **pair})\n    write_report(output, rows, wrong_examples)\n    summary = {"status": "completed_frozen_model_sweep", "samples": sum(r["samples"] for r in rows),\n               "grids": len(rows), "checkpoint_epoch": checkpoint["epoch"],\n               "checkpoint_sha256": expected_checkpoint_sha256, "calibration_manifest_sha256": digest(directory / "manifest.json"),\n               "pixel_sha256": manifest["pixel_sha256"], "environment": T.environment(device),\n               "original_training_environment": checkpoint["config"]["environment"], "precision": "fp32_no_autocast",\n               "clean_control_accuracy": rows[0]["macro_accuracy"], "clean_control_matches_archived": rows[0]["errors"] == 0,\n               "evaluator_sha256": digest(__file__), "human_readability": "not_measured",\n               "retrained_baseline": "not_run", "test_evaluated": False,\n               "interpretation": "New-corruption evaluation of the old model; not final-test or matched-training evidence."}\n    write_json(output / "summary.json", summary)\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest="command", required=True)\n    build = sub.add_parser("generate")\n    build.add_argument("--data", type=Path, default=ROOT / "training/data/baseline81-v2")\n    build.add_argument("--output", type=Path, required=True)\n    build.add_argument("--font", type=Path, help="Also build all 54 pair previews using this font")\n    run = sub.add_parser("evaluate")\n    run.add_argument("--data", type=Path, required=True)\n    run.add_argument("--checkpoint", type=Path, required=True)\n    run.add_argument("--output", type=Path, required=True)\n    run.add_argument("--device", choices=["auto", "cpu", "cuda"], default="auto")\n    run.add_argument("--batch-size", type=int, default=128)\n    args = parser.parse_args()\n    if args.command == "generate":\n        manifest, audit = generate(args.data, args.output)\n        if args.font:\n            make_gallery(args.output, args.font)\n        print(json.dumps({"output": str(args.output), "samples": manifest["samples"], "audit": audit}, ensure_ascii=False, indent=2))\n    else:\n        if args.batch_size <= 0:\n            parser.error("batch-size must be positive")\n        print(json.dumps(evaluate(args.data, args.checkpoint, args.output, args.batch_size, args.device), ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'}

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
checkpoint = DRIVE_ROOT / "runs/baseline81_s42_v1/best.pt"
if not checkpoint.is_file():
    raise FileNotFoundError(f"缺少原模型：{checkpoint}")
if importlib.metadata.version("Pillow") != "12.3.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "Pillow==12.3.0"], check=True)

def file_digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if file_digest(checkpoint) != "8d65412bf6065253ebce3ca5b0c542788b71e0da391cc335a815f980d828a106":
    raise ValueError("best.pt 与归档的第 38 轮模型不一致")
WORK.mkdir(parents=True, exist_ok=True)
for name, source in SOURCES.items():
    target = WORK / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

bundle_hash = "e477bdd4894e6b585a2149d6f22db3948e5d6e20075d532fded5171d1b57dd35"
candidates = [DRIVE_ROOT / "hanzi81_colab_bundle.zip", DRIVE_ROOT / "packages" / (bundle_hash + ".zip")]
archive_path = next((p for p in candidates if p.is_file()), None)
if archive_path is None:
    raise FileNotFoundError(f"找不到原始数据包，请检查：{candidates}")
source = WORK / "training/data/baseline81-v2"
source.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    # Only extract development data and metadata. train/test arrays stay in the ZIP.
    for name in ("manifest.json", "labels.json", "conditions.json", "val_graph.npz"):
        (source / name).write_bytes(archive.read("training/data/baseline81-v2/" + name))
calibration = WORK / "training/data/degradation81-calibration-v1"
if not calibration.exists():
    subprocess.run([sys.executable, "-u", "-m", "training.degradation", "generate",
                    "--data", str(source), "--output", str(calibration)], cwd=WORK, check=True)
manifest = json.loads((calibration / "manifest.json").read_text(encoding="utf-8"))
if manifest["pixel_sha256"] != EXPECTED_PIXELS:
    raise ValueError("标定像素与本地已检查版本不同，停止推理并检查 Pillow / 输入文件")
for name, expected in manifest["files"].items():
    if file_digest(calibration / name) != expected:
        raise ValueError(f"标定数据文件损坏：{name}")
print("准备完成：", manifest["samples"], "张 /", len(manifest["grids"]), "档；原模型未更新。")


In [ ]:
#@title ② 执行 25 档推理并保存新报告
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json
import shutil
import subprocess
import sys
from IPython.display import Image, display
from google.colab import files

DEVICE = "auto" #@param ["auto", "cuda", "cpu"]
BATCH_SIZE = 128 #@param {type:"integer"}
DRIVE_ROOT = Path("/content/drive/MyDrive/hanzi81")
WORK = Path("/content/hanzi81_degradation_phase1_v1")
if not (WORK / "training/data/degradation81-calibration-v1/manifest.json").is_file():
    raise FileNotFoundError("请先运行第①格准备标定集")
stamp = datetime.now(timezone(timedelta(hours=8))).strftime("%Y%m%dT%H%M%S_%f")
local_report = WORK / "reports" / stamp
subprocess.run([sys.executable, "-u", "-m", "training.degradation", "evaluate",
    "--data", str(WORK / "training/data/degradation81-calibration-v1"),
    "--checkpoint", str(DRIVE_ROOT / "runs/baseline81_s42_v1/best.pt"),
    "--output", str(local_report), "--batch-size", str(BATCH_SIZE), "--device", DEVICE], cwd=WORK, check=True)
summary = json.loads((local_report / "summary.json").read_text(encoding="utf-8"))
if summary["status"] != "completed_frozen_model_sweep" or summary["samples"] != 32400 or summary["grids"] != 25:
    raise ValueError("评估未完成全部 25 档，未标记阶段完成")
shutil.copy2(WORK / "training/data/degradation81-calibration-v1/manifest.json", local_report / "calibration_manifest.json")
shutil.copy2(WORK / "training/data/degradation81-calibration-v1/audit.json", local_report / "calibration_audit.json")
shutil.copy2(WORK / "training/degradation.py", local_report / "degradation.py")
drive_report = DRIVE_ROOT / "analyses/baseline81_s42_v1/07_degradation_phase1" / stamp
shutil.copytree(local_report, drive_report)
archive = shutil.make_archive(str(WORK / "hanzi81_phase1_results"), "zip", local_report)
shutil.copy2(archive, drive_report / "hanzi81_phase1_results.zip")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("完整报告已保存：", drive_report)
display(Image(filename=str(local_report / "accuracy_curves.png")))
files.download(archive)
